# Checkpoint A: a clean agent on a track it was not tuned for

This checkpoint closes **Part 1**.

**The brief.**
Your Unit 7 agent is fast on `test-circuit`.
Race day is on a track you have never driven: a variant of `test-circuit`, every corner a little tighter or looser, every straight a little narrower or wider.
You may tune on four other variants of the same family.
You may race the checkpoint track once.

The tracks, all generated by racekit and loaded by name with `load_track`:

- **the training family**, five tracks you may race as often as you like: `test-circuit`, `test-circuit-seed1`, `test-circuit-seed2`, `test-circuit-seed3` and `test-circuit-seed4`;
- **the checkpoint track**, `test-circuit-seed2026`, raced **once**, on race day (Question 6).

The variants differ from `test-circuit` in shape, and the carla-like car may not grip the same on every track, as no two real circuits grip the same.

## The rules

- **One global setting.** You tune `A_LAT` only, on Unit 7's grid (steps of 0.2 m/s²).
- **Everything else is fixed at the Unit 7 reference's values:** `A_DEC = 5.0`, `PP_K = 0.5`, `PROFILE_LEAD_S = 0.3`, and `GRIP` all 1.0.
  Your Unit 7 file may ship the per-section `GRIP` vector you accepted in Unit 7; reset it, because a section is a sixth of a different lap on every track.
  The setup cell's `setting` function writes these fixed values into every variant for you.
- **The clean rule** (Unit 7): a race is clean when it finished, with 0 collisions, and its max |cross-track error| over the driving rows is at most 2 m.
  A setting is clean on a track when it is clean on **race seeds 0–4**.
- **Every race** uses `carla_like(noise=True)` and `max_seconds=450` (a clean lap takes about 320 s).
- **The target:** on the checkpoint track, the median race time over seeds 0–4 is at most **330 s**, with every race clean.
- **No tuning on the checkpoint track.** Loading it and planning on it is allowed; racing it before race day is not.
  Freeze your setting first (Question 5), then race the checkpoint track once, on five seeds (Question 6).

**Your submission** is your own Unit 7 submission.
The setup cell points `SUBMISSION` at `agents/unit-07.py`, the book's Unit 7 reference, as a stand-in: replace that path with your own Unit 7 file before you begin.

Run the setup cell first.
Each question names the variables your answer must define; fill in every `None`, and write each written answer in the markdown cell under its prompt.
Run the questions in order: later questions reuse earlier functions and results.
A race takes a few seconds; no cell should race more than one track, so each cell stays short.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.runner import run_race
from racekit.tracks import load_track

AGENTS = Path("../../projects/racekit/agents")
SUBMISSION = AGENTS / "unit-07.py"  # a stand-in: replace it with the path of YOUR Unit 7 submission
REFERENCE = AGENTS / "unit-07.py"  # the book's Unit 7 reference: Question 4's fallback only; never your submission
workdir = Path(mkdtemp(prefix="roar-checkpoint-a-"))

TRAINING = ("test-circuit", "test-circuit-seed1", "test-circuit-seed2", "test-circuit-seed3", "test-circuit-seed4")
CHECKPOINT_TRACK = "test-circuit-seed2026"  # race it once, in Question 6, and never before
SEEDS = (0, 1, 2, 3, 4)  # the clean rule's five race seeds
GRID_STEP = 0.2  # m/s^2: Unit 7's A_LAT grid
FIXED = {"A_DEC": 5.0, "PP_K": 0.5, "PROFILE_LEAD_S": 0.3, "GRIP": (1.0,) * 6}  # the Unit 7 reference's values
A_ACC, TOP_SPEED = 4.0, 45.0  # m/s^2 and m/s: the reference's forward pass and top speed (use yours if your file differs)
MAX_SECONDS = 450.0  # s: every race's time cap
CLEAN_OFFSET = 2.0  # m: the clean rule's limit on the max |cross-track error|
TARGET_S = 330.0  # s: race day's target median
UNIT_6_TIME = 420.40  # s: the Unit 6 agent's median on the checkpoint track, seeds 0-4 (given)

race_log = []  # every race you run, as (track, seed): your evidence of the sweep


def setting(a_lat, base=SUBMISSION):
    # Unit 1's idiom: a copy of base (SUBMISSION unless you say otherwise) with A_LAT set and the other
    # knobs fixed, written into workdir.
    a_lat = round(a_lat, 1)
    constants = {"A_LAT": a_lat, **FIXED}
    name = f"a-lat-{a_lat}.py" if base == SUBMISSION else f"reference-a-lat-{a_lat}.py"
    return write_variant(base, workdir / name,
                         {key: repr(value) for key, value in constants.items()})


def race(source, track, seed):
    # Every race of this checkpoint: the carla-like car with its noise, one seed, the 450 s cap.
    race_log.append((track, seed))
    return run_race(source, track, car=carla_like(noise=True), seed=seed, max_seconds=MAX_SECONDS)


def max_offset(result):
    # The largest |cross-track error| over the driving rows (held == 0).
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def clean(result):
    # Unit 7's clean rule: finished, 0 collisions, and max |cross-track error| at most 2 m.
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET

## Question 1 - Know the family

Before you race anything, look at what you will be tuning on.
Load the five training tracks with `load_track`; each has waypoints `track.x`, `track.y` (one lap, closed: the last waypoint joins waypoint 0), its length `track.length` (m), and the curvature the generator built it with, `track.curvature` (1/m).

For each track, compute from the waypoints yourself:

- the lap length, the sum of the n straight-line segment lengths from waypoint i to waypoint i + 1 (the last segment closes the lap);
- κ_i, Unit 2's Menger curvature of the waypoints (i − 1, i, i + 1), cyclic: 4 × the triangle's area divided by the product of its three sides;
- the tightest radius, 1 / max |κ_i|.

Then compare your curvature with the stored one.

Define:

- `tracks`: a dict from each name in `TRAINING` to its loaded track;
- `menger(xs, ys)`: a function returning a NumPy array of the cyclic Menger curvature at every waypoint (the sign does not matter here, only |κ|);
- `lap_length`: a dict from name to your lap length (m);
- `length_gap`: a dict from name to |your lap length − `track.length`| (m);
- `tightest_radius`: a dict from name to 1 / max |κ_i| from your curvature (m);
- `stored_tightest_radius`: a dict from name to 1 / max |`track.curvature`| (m);
- `curvature_gap`: a dict from name to the largest | |κ_i| − |`track.curvature[i]`| | over the waypoints (1/m);
- `fig`, `ax`: one plot of the five centrelines (one line per track, labelled with its name), with equal axis scales, axis labels, a legend and a title.

In [ ]:
tracks = {}  # TODO


def menger(xs, ys):
    return None  # TODO: a NumPy array of n curvatures (1/m), cyclic


lap_length = {}  # TODO
length_gap = {}  # TODO
tightest_radius = {}  # TODO
stored_tightest_radius = {}  # TODO
curvature_gap = {}  # TODO
fig, ax = None, None  # TODO

**Written answer.**
Which training track has the tightest corner, and how far apart are the five tracks' lengths?
From your plot, in one or two sentences: would you expect a setting that is clean on `test-circuit` to be clean on all of them, and why?

*Your answer here.*

## Question 2 - Plan a lap on paper

Write Unit 7's forward–backward speed profile as a function, so you can plan any track before you race it.
For a closed lap of n waypoints:

1. the cap of every waypoint, min(`top_speed`, √(`a_lat` / |κ_i|)) where |κ_i| > 1e-9, and `top_speed` elsewhere (κ_i your `menger` from Question 1);
2. the backward pass, twice round the lap: for i from n − 1 **down** to 0, v_i = min(v_i, √(v_{(i+1) mod n}² + 2·`a_dec`·Δs_i));
3. then the forward pass, twice round the lap: for i from 0 **up** to n − 1, v_{(i+1) mod n} = min(v_{(i+1) mod n}, √(v_i² + 2·`a_acc`·Δs_i)).

Δs_i is the length of the segment from waypoint i to waypoint i + 1 (the last segment closes the lap).
The planned lap time is the sum over the n segments of Δs_i / ((v_i + v_{(i+1) mod n}) / 2).

Plan each training track at the Unit 7 reference's `A_LAT = 4.6`, with `A_DEC` from `FIXED`, `A_ACC` and `TOP_SPEED`.
Then check every profile against the lesson's two passes:

- **the cap pass:** no waypoint is planned above its cornering cap, v_i ≤ cap_i (to within 1e-9);
- **the backward pass:** no step brakes harder than `A_DEC`, v_i² ≤ v_{(i+1) mod n}² + 2·`A_DEC`·Δs_i (to within 1e-9), at every i including the wrap-around.

Define:

- `lap_profile(xs, ys, a_lat, a_dec, a_acc, top_speed)`: a function returning `(profile, cap, ds)`, three NumPy arrays of n floats;
- `planned_lap_time(profile, ds)`: a function returning the planned time of one lap (s);
- `planned_4_6`: a dict from each name in `TRAINING` to its planned lap time at `A_LAT = 4.6` (s);
- `cap_pass_ok`: a dict from name to `True` when the cap pass check holds at every waypoint;
- `backward_pass_ok`: a dict from name to `True` when the backward pass check holds at every waypoint;
- `fig`, `ax`: one plot of `test-circuit-seed1`'s cap (at most `TOP_SPEED`) and profile against the distance along the lap (`track.s`), with axis labels, a legend and a title.

In [ ]:
def lap_profile(xs, ys, a_lat, a_dec, a_acc, top_speed):
    return None, None, None  # TODO: (profile, cap, ds)


def planned_lap_time(profile, ds):
    return None  # TODO: seconds for one lap


planned_4_6 = {}  # TODO
cap_pass_ok = {}  # TODO
backward_pass_ok = {}  # TODO
fig, ax = None, None  # TODO

**Written answer.**
A race is three laps, so a planned lap is not a race time.
In one or two sentences: name one thing the plan assumes that the race will not give it.

*Your answer here.*

## Question 3 - Tune on one track

Do what Unit 7 did, on `test-circuit` alone.

First write the clean rule on five seeds as a function.
`five_seed_check(source, track)` races `source` on `track` with `race`, on the seeds in `SEEDS` in order, and **stops at the first race that is not clean** (one failing race is enough to call the setting not clean, and the races after it would only cost time).
It returns `(all_clean, results)`: `all_clean` is `True` when all five races were clean, and `results` is the list of the races it ran.

Then find `test-circuit`'s edge on the grid: start at your Unit 7 submission's `A_LAT` (the reference ships 4.6) and step **up** by `GRID_STEP`, one `setting(a_lat)` variant per value, until a value is not clean.
The edge is the last clean value; the margin rule (Unit 7) ships one grid step inside it.
(If your starting value is itself not clean, step down instead until you find a clean one.)

Race only `test-circuit` in this cell.

Define:

- `five_seed_check(source, track)`: the function;
- `one_track_sweep`: a dict from each `A_LAT` you raced (rounded to 1 decimal) to `(all_clean, median)`, with `median` the median race time over seeds 0–4 (s) when `all_clean`, and `None` otherwise;
- `one_track_edge`: the highest clean value you found;
- `one_track_margin`: the margin rule's value, `one_track_edge - GRID_STEP`, rounded to 1 decimal.

In [ ]:
def five_seed_check(source, track):
    return None, []  # TODO: (all_clean, results)


one_track_sweep = {}  # TODO: race test-circuit only
one_track_edge = None  # TODO
one_track_margin = None  # TODO

## Question 4 - Check the family

A setting tuned on one track has only been tested on that track.
Race `one_track_margin` on the four variants, `test-circuit-seed1` to `test-circuit-seed4`, with `five_seed_check`: one track per cell, below.

Define:

- `family_check`: a dict from each variant's name to the `(all_clean, results)` that `five_seed_check(setting(one_track_margin), name)` returned;
- `failing_track`: the name of the first variant (in the order of `TRAINING`) on which `one_track_margin` is not clean (`None` if it is clean on all four);
- `failing_race`: the race that was not clean on `failing_track` (the last race in its `results`).

**If your `one_track_margin` is clean on all four variants**, your own agent gives you no failure to study, so study the book's: race `setting(4.6, REFERENCE)` (the Unit 7 reference at its shipped A_LAT) on `test-circuit-seed1` with `five_seed_check`, in one more cell, and use that track and its failing race as `failing_track` and `failing_race` below. Your own `family_check` still records your clean result; say in your written answer which agent the failure came from.

In [ ]:
family_check = {}  # TODO: a dict, filled one track per cell
# TODO: race one_track_margin on "test-circuit-seed1" only

In [ ]:
# TODO: race one_track_margin on "test-circuit-seed2" only

In [ ]:
# TODO: race one_track_margin on "test-circuit-seed3" only

In [ ]:
# TODO: race one_track_margin on "test-circuit-seed4" only

Now look at the failing race.
Its first collision happens at `failing_race.respawn_times[0]` (Unit 1).
Plot what the car did in the 10 s before it.

Define:

- `first_collision_t`: the time of the failing race's first collision (s);
- `window`: a boolean array over the failing race's telemetry rows, `True` for the rows with `first_collision_t - 10 <= t <= first_collision_t`;
- `fig`, `axes`: one figure with two plots, one above the other with a shared time axis: the cross-track error (`offset`, m) against `t`, and the speed (`v`, m/s) against `t`, over `window`; with axis labels and a title naming the track, the `A_LAT` and the race seed;
- `offset_every_half_second`: a list of the cross-track error (m) at the rows every 0.5 s through `window`, from its first row to its last (a telemetry row is one control tick of 0.05 s, so every 10th row).

In [ ]:
first_collision_t = None  # TODO
window = None  # TODO
fig, axes = None, None  # TODO
offset_every_half_second = []  # TODO

**Written answer.**
Answer in three parts.

1. **Describe** what the car did in those 10 s: where it was (on a corner or a straight, and after what), how fast it was going, and what the cross-track error did, with numbers from your plot.
2. **Propose a cause** in the terms of Units 5–6: for example the pure-pursuit lookahead at that speed, the grip the car had there, or the corner before.
   You are graded on the description and the reasoning, not on finding one particular mechanism.
3. **Name one race** (one setting, on one training track, on one seed) that would test your proposed cause, and say what result would support it and what result would count against it.
   You do not have to run it.
   The race is a diagnostic, not your submission, so it may change any knob, a fixed one included; your submission still tunes `A_LAT` only.

*Your answer here.*

## Question 5 - Tune on the family

Now tune on the whole family: a setting counts as clean only when it is clean on **all five** training tracks.
Find the highest grid value that is, apply the margin rule, and freeze it.

Use what you have already raced: Questions 3 and 4 tell you which values are already not clean somewhere.
Step down the grid from there, racing one track per cell, below; stop racing a value as soon as one track is not clean.
If a value is not clean somewhere, step down one grid step and run the five cells again from `test-circuit`, keeping every result in `family_sweep`.

Define:

- `family_sweep`: a dict from `(a_lat, name)` (with `a_lat` rounded to 1 decimal) to `(all_clean, median)` for every value and track you raced in this question, `median` as in Question 3;
- `family_edge`: the highest grid value clean on all five training tracks;
- `FROZEN_A_LAT`: the margin rule's value, `family_edge - GRID_STEP`, rounded to 1 decimal;
- `FROZEN_SUBMISSION`: `setting(FROZEN_A_LAT)`, the file you will race on race day;
- `frozen_report`: `check_submission(FROZEN_SUBMISSION)`, with `frozen_report.ok` `True`.

After this question your setting is frozen: nothing about it may change once you have raced the checkpoint track.

In [ ]:
family_sweep = {}  # TODO: a dict, filled one track per cell
# TODO: race your candidate value on "test-circuit" only

In [ ]:
# TODO: race your candidate value on "test-circuit-seed1" only

In [ ]:
# TODO: race your candidate value on "test-circuit-seed2" only

In [ ]:
# TODO: race your candidate value on "test-circuit-seed3" only

In [ ]:
# TODO: race your candidate value on "test-circuit-seed4" only

In [ ]:
family_edge = None  # TODO
FROZEN_A_LAT = None  # TODO
FROZEN_SUBMISSION = None  # TODO
frozen_report = None  # TODO

**Written answer.**
Unit 7 already used a margin of one grid step on `test-circuit`.
In two or three sentences: why was that margin not enough for the family, and what does the margin you applied here protect you against that the family sweep cannot?

*Your answer here.*

## Question 6 - Race day

Your setting is frozen.
**First plan, then race.**

Load the checkpoint track (allowed) and plan it with Question 2's `lap_profile` and `planned_lap_time`, at `FROZEN_A_LAT` and the fixed knobs.
Also plan it one grid step higher, at `FROZEN_A_LAT + GRID_STEP` (planning is allowed; racing that value here is not): the difference is what the margin costs on paper.

Define:

- `checkpoint_track`: `load_track(CHECKPOINT_TRACK)`;
- `planned_lap_frozen`: the planned lap time at `FROZEN_A_LAT` (s);
- `planned_race_frozen`: the planned time of a three-lap race, `3 * planned_lap_frozen` (s);
- `planned_margin_cost`: the planned lap time at `FROZEN_A_LAT` minus the planned lap time at `FROZEN_A_LAT + GRID_STEP`, times 3 (s per race).

In [ ]:
checkpoint_track = None  # TODO
planned_lap_frozen = None  # TODO
planned_race_frozen = None  # TODO
planned_margin_cost = None  # TODO

Now race `FROZEN_SUBMISSION` on the checkpoint track, **once**, on every seed in `SEEDS`.
Race all five seeds even if one is not clean: race day is the whole result.
Run this cell once.

Define:

- `race_day`: the list of the five races, in seed order (race each with `race`, not with `five_seed_check`, which stops early);
- `race_day_times`: the list of their race times (s);
- `race_day_clean`: `True` when all five races are clean;
- `race_day_median`, `race_day_spread`: the median and the spread (max − min) of the times (s);
- `within_target`: `True` when `race_day_clean` and `race_day_median <= TARGET_S`;
- `race_minus_plan`: `race_day_median - planned_race_frozen` (s);
- `gain_over_unit_6`: `UNIT_6_TIME - race_day_median` (s);
- `checkpoint_races`: the number of entries in `race_log` on `CHECKPOINT_TRACK` (it should be exactly 5).

In [ ]:
race_day = []  # TODO: race CHECKPOINT_TRACK only, once, seeds 0-4
race_day_times = []  # TODO
race_day_clean = None  # TODO
race_day_median, race_day_spread = None, None  # TODO
within_target = None  # TODO
race_minus_plan = None  # TODO
gain_over_unit_6 = None  # TODO
checkpoint_races = None  # TODO

**Written answer.**
In three to five sentences, report race day: the median and spread, the clean rule and the target.
Compare the race with your plan: is the race faster or slower than the plan, by how much, and why? Is the gap what you would expect from Question 2's answer?
Compare it with the Unit 6 agent's 420.40 s: what did speed planning buy?
And what did the margin cost, on paper (`planned_margin_cost`), against what it bought you on a track you had never raced?

*Your answer here.*